<a href="https://colab.research.google.com/github/ChristopherMwanginjoroge/deep-learning/blob/main/BGans.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import torch
import torch.nn as nn
from torchvision import datasets,transforms
from torch.utils.data import DataLoader

transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,),(0.5,))

])

train_data=datasets.MNIST(root='data',train=True,download=True,transform=transform)

dataloader=DataLoader(train_data,batch_size=64,shuffle=True)


class Generator(nn.Module):
  def __init__(self):
    super(Generator,self).__init__()

    self.fc1=nn.Linear(100,256)
    self.fc2=nn.Linear(256,512)
    self.fc3=nn.Linear(512,1024)
    self.fc4=nn.Linear(1024,28 * 28)



  def forward(self,x):
    x=torch.relu(self.fc1(x))
    x=torch.relu(self.fc2(x))
    x=torch.relu(self.fc3(x))
    x=torch.tanh(self.fc4(x))

    x=x.view(-1,1,28 * 28)

    return x




class Discriminator(nn.Module):
  def __init__(self):
    super(Discriminator,self).__init__()

    self.fc1=nn.Linear(1024, 28*28)
    self.fc2=nn.Linear(256,128)
    self.fc3=nn.Linear(128,1)

    self.dropout = nn.Dropout(0.3)

  def forward(self, x):
        # Flatten the image
    x = x.view(-1, 28 * 28)


    x = torch.leaky_relu(self.fc1(x), 0.2) # Leaky ReLU is standard for GAN Discriminators

    x = self.dropout(x)
    x = torch.leaky_relu(self.fc2(x), 0.2)


    x = self.dropout(x)
    x = torch.leaky_relu(self.fc3(x), 0.2)
    x = self.dropout(x)
        # Sigmoid to output a probability between 0 (Fake) and 1 (Real)

    x = torch.sigmoid(self.fc4(x))
    return x

In [13]:
generator = Generator()
discriminator = Discriminator()

print("Generator Architecture:")
print(generator)
print("\nDiscriminator Architecture:")
print(discriminator)

Generator Architecture:
Generator(
  (fc1): Linear(in_features=100, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=512, bias=True)
  (fc3): Linear(in_features=512, out_features=1024, bias=True)
  (fc4): Linear(in_features=1024, out_features=784, bias=True)
)

Discriminator Architecture:
Discriminator(
  (fc1): Linear(in_features=1024, out_features=784, bias=True)
  (fc2): Linear(in_features=256, out_features=128, bias=True)
  (fc3): Linear(in_features=128, out_features=1, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
)


In [14]:
import torch.optim as optim

g_optimizer=optim.Adam(generator.parameters(),lr=0.0002)
d_optimizer=optim.Adam(discriminator.parameters(),lr=0.0002)
criterion=nn.BCELoss()

epochs=50

for epoch in range(epochs):
  for real_images,_ in dataloader:
    batch_size=real_images.size(0)
    real_labels=torch.ones(batch_size,1)
    fake_labels=torch.zeros(batch_size,1)

    d_optimizer.zero_grad()

    real_outputs=discriminator(real_images)

    d_loss_real=criterion(real_outputs,real_labels)

    noise=torch.randn(batch_size,100)

    fake_images=generator(noise)

    fake_outputs=discriminator(fake_images.detach())

    d_loss_fake=criterion(fake_outputs,fake_labels)

    d_loss=d_loss_real+d_loss_fake

    d_loss.backward()

    d_optimizer.step()


    g_optimizer.zero_grad()

    noise = torch.randn(batch_size, 100)
    fake_images = generator(noise)
    fake_outputs = discriminator(fake_images)
    g_loss = criterion(fake_outputs, real_labels)
    g_loss.backward()
    g_optimizer.step()

  if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{epochs}] | Cop Loss (D): {d_loss.item():.4f} | Counterfeiter Loss (G): {g_loss.item():.4f}")

print("\nTraining complete! The Counterfeiter has finished practicing.")



AttributeError: module 'torch' has no attribute 'leaky_relu'